# NF Sweep Inspection

Use this notebook after submitting `scripts/slurm/train_nf_sweep_array.sbatch` and `scripts/slurm/sample_nf_sweep_array.sbatch`.

It checks, in order:

- which sweep runs/configs/checkpoints/samples exist
- training loss and learning-rate curves
- post-hoc EMA profile geometry
- generated sample images versus real training slices
- one-point field histograms
- 2D power spectra with real reference bands
- metric tables across variants and EMA targets
- optional SSCD copy/generalization diagnostics

Most cells are safe on a login node. Cells that load many samples or run SSCD should be run after sample files exist, and SSCD is set to CPU by default to avoid CUDA/NVRTC issues on notebooks.

In [ ]:
from __future__ import annotations

import json
import math
import os
import re
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", Path.cwd())).resolve()
if not (PROJECT_DIR / "scripts").exists():
    PROJECT_DIR = Path("/home/jiamingp/diffusion_models_repo")

# Prefer the verified git checkout. Do not trust an inherited stale COSMODIFF_DIR.
COSMODIFF_DIR = Path(os.environ.get(
    "COSMODIFF_DIR_OVERRIDE",
    "/home/jiamingp/Diffusion_model/cosmo_diffusion_normalization_fixes_git",
)).resolve()
if COSMODIFF_DIR.exists() and str(COSMODIFF_DIR) not in sys.path:
    sys.path.insert(0, str(COSMODIFF_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_npy, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 180})

print("project:", PROJECT_DIR)
print("cosmodiff_dir:", COSMODIFF_DIR, "exists=", COSMODIFF_DIR.exists())

In [ ]:
CONFIG_DIR = PROJECT_DIR / "local" / "nf_sweep" / "configs"
MANIFEST_PATH = PROJECT_DIR / "local" / "nf_sweep" / "manifest.json"
CHECKPOINT_ROOT = Path("/scratch/huterer_root/huterer0/jiamingp/saved_runs/nf_sweep")
SAMPLE_ROOT = PROJECT_DIR / "results" / "nf_sweep" / "samples"
OUTPUT_DIR = PROJECT_DIR / "results" / "nf_sweep" / "inspection"
CACHE_DIR = PROJECT_DIR / "results" / "cache" / "nf_sweep_inspection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 123
EMA_LABELS = ["raw", "ema0p03", "ema0p05", "ema0p08", "ema0p13", "ema0p20"]
EMA_VALUES = {"raw": np.nan, "ema0p03": 0.03, "ema0p05": 0.05, "ema0p08": 0.08, "ema0p13": 0.13, "ema0p20": 0.20}
PREFERRED_SAMPLE_LABEL = "ema0p13"

# Keep these modest on login nodes. Increase after things look sane.
MAX_RAW_REAL_CUBES = 32      # 32 raw cubes -> 1024 2D slices with zthin=4
MAX_REAL_PLOT = 64
MAX_REAL_PK = 256
MAX_GENERATED_PLOT = 64
MAX_GENERATED_PK = 128
PK_NBINS = 25

RUN_SSCD = False
SSCD_PATH = Path(os.environ.get("SSCD_PATH", Path.home() / ".cache" / "torch" / "hub" / "sscd_disc_mixup.torchscript.pt"))
SSCD_DEVICE = "cpu"
SSCD_BATCH_SIZE = 8
SSCD_THRESHOLD = 0.60
SSCD_RENDER_MODE = "fixed"

print("manifest:", MANIFEST_PATH, "exists=", MANIFEST_PATH.exists())
print("checkpoint_root:", CHECKPOINT_ROOT)
print("sample_root:", SAMPLE_ROOT)
print("sscd:", SSCD_PATH, "exists=", SSCD_PATH.exists())

## Run Discovery

This reads `local/nf_sweep/manifest.json`, config YAMLs, checkpoint folders, and sample files. It can be rerun while jobs are still finishing.

In [ ]:
def latest_checkpoint(run_name: str) -> Path | None:
    ckpt_dir = CHECKPOINT_ROOT / f"{run_name}_checkpoints"
    if not ckpt_dir.exists():
        return None
    candidates = sorted(ckpt_dir.glob("checkpoint-epoch-*"))
    return candidates[-1] if candidates else None


def checkpoint_epoch(path: Path | None) -> int | None:
    if path is None:
        return None
    m = re.search(r"checkpoint-epoch-(\d+)$", path.name)
    return int(m.group(1)) if m else None


def sample_path(run_name: str, ema_label: str = PREFERRED_SAMPLE_LABEL) -> Path:
    return SAMPLE_ROOT / f"{run_name}_seed{SEED}_{ema_label}.npy"


def sample_shape(path: Path) -> tuple[int, ...] | None:
    if not path.exists():
        return None
    try:
        return tuple(np.load(path, mmap_mode="r").shape)
    except Exception:
        return None

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing {MANIFEST_PATH}. Run: python scripts/prepare_nf_sweep_configs.py --project-dir $PWD"
    )

manifest = json.loads(MANIFEST_PATH.read_text())
run_rows = []
for row in manifest:
    run_name = row["run_name"]
    ckpt = latest_checkpoint(run_name)
    config_path = PROJECT_DIR / row["config"]
    sample_exists = {label: sample_path(run_name, label).exists() for label in EMA_LABELS}
    run_rows.append({
        **row,
        "config_path": str(config_path),
        "config_exists": config_path.exists(),
        "checkpoint_path": str(ckpt) if ckpt else None,
        "checkpoint_epoch": checkpoint_epoch(ckpt),
        "has_any_sample": any(sample_exists.values()),
        **{f"has_{label}": exists for label, exists in sample_exists.items()},
    })

run_df = pd.DataFrame(run_rows)
display(run_df[[
    "run_name", "arch", "variant_tag", "beta_schedule", "prediction_type",
    "sigma_log_normal", "min_snr_gamma", "config_exists", "checkpoint_epoch", "has_any_sample",
    *[f"has_{label}" for label in EMA_LABELS],
]])

In [ ]:
# Check that the configs contain the intended knobs.
config_rows = []
for row in run_df.to_dict("records"):
    path = Path(row["config_path"])
    if not path.exists():
        continue
    cfg = yaml.safe_load(path.read_text())
    ns = cfg["noise_scheduler"]["kwargs"]
    train = cfg["train"]
    config_rows.append({
        "run_name": row["run_name"],
        "arch": row["arch"],
        "variant": row["variant_tag"],
        "beta_schedule": ns.get("beta_schedule"),
        "prediction_type": ns.get("prediction_type"),
        "rescale_betas_zero_snr": ns.get("rescale_betas_zero_snr", False),
        "timestep_spacing": ns.get("timestep_spacing", None),
        "sigma_log_normal": train.get("sigma_log_normal"),
        "min_snr_gamma": train.get("min_snr_gamma"),
        "ema_sigma_rels": train.get("ema_sigma_rels"),
        "ema_update_every": train.get("ema_update_every"),
    })
config_df = pd.DataFrame(config_rows)
display(config_df)

## Training Curves

The training script writes `metrics.json` inside checkpoint folders and `metrics_epoch_*.json` at the run root. These cells read the newest metrics available per run.

In [ ]:
def metric_candidates(run_name: str) -> list[Path]:
    root = CHECKPOINT_ROOT / f"{run_name}_checkpoints"
    paths = []
    paths.extend(sorted(root.glob("metrics_epoch_*.json")))
    paths.extend(sorted(root.glob("checkpoint-epoch-*/metrics.json")))
    return [p for p in paths if p.exists()]


def load_metrics(run_name: str) -> tuple[dict[str, Any] | None, Path | None]:
    paths = metric_candidates(run_name)
    if not paths:
        return None, None
    # Prefer newest by mtime because checkpoint metrics contain cumulative history.
    path = max(paths, key=lambda p: p.stat().st_mtime)
    try:
        return json.loads(path.read_text()), path
    except Exception as exc:
        print("failed metrics", run_name, path, exc)
        return None, path

metrics_by_run = {}
metric_rows = []
for row in run_df.to_dict("records"):
    metrics, path = load_metrics(row["run_name"])
    if metrics is None:
        metric_rows.append({
            "run_name": row["run_name"],
            "arch": row["arch"],
            "variant": row["variant_tag"],
            "has_metrics": False,
        })
        continue
    metrics_by_run[row["run_name"]] = metrics
    loss = np.asarray(metrics.get("loss", []), dtype=float)
    epoch_loss = np.asarray(metrics.get("epoch_loss", []), dtype=float)
    epoch_lr = np.asarray(metrics.get("epoch_lr", []), dtype=float)
    metric_rows.append({
        "run_name": row["run_name"],
        "arch": row["arch"],
        "variant": row["variant_tag"],
        "has_metrics": True,
        "metrics_path": str(path),
        "n_batch_steps_logged": len(loss),
        "n_epochs_logged": len(epoch_loss),
        "latest_epoch_loss": float(epoch_loss[-1]) if len(epoch_loss) else np.nan,
        "best_epoch_loss": float(np.nanmin(epoch_loss)) if len(epoch_loss) else np.nan,
        "latest_lr": float(epoch_lr[-1]) if len(epoch_lr) else np.nan,
    })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.sort_values(["arch", "variant"]))
metrics_csv = OUTPUT_DIR / "nf_sweep_metrics_status.csv"
metrics_df.to_csv(metrics_csv, index=False)
print("wrote", metrics_csv)

In [ ]:
def moving_average(x: np.ndarray, window: int) -> np.ndarray:
    if len(x) == 0 or window <= 1:
        return x
    window = min(window, len(x))
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="valid")

for arch in sorted(run_df["arch"].unique()):
    rows = run_df[run_df["arch"] == arch]
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for _, row in rows.iterrows():
        name = row["run_name"]
        metrics = metrics_by_run.get(name)
        if not metrics:
            continue
        label = row["variant_tag"]
        loss = np.asarray(metrics.get("loss", []), dtype=float)
        epoch_loss = np.asarray(metrics.get("epoch_loss", []), dtype=float)
        epoch_lr = np.asarray(metrics.get("epoch_lr", []), dtype=float)
        if len(loss):
            ma = moving_average(loss, 200)
            x = np.arange(len(ma)) + max(0, len(loss) - len(ma))
            axes[0].plot(x, ma, label=label, lw=1.5)
        if len(epoch_loss):
            axes[1].plot(np.arange(len(epoch_loss)), epoch_loss, marker="o", ms=2, label=label)
        if len(epoch_lr):
            axes[2].plot(np.arange(len(epoch_lr)), epoch_lr, marker="o", ms=2, label=label)
    axes[0].set_title(f"{arch}: batch loss, moving avg")
    axes[0].set_xlabel("optimizer step")
    axes[0].set_ylabel("MSE loss")
    axes[0].set_yscale("log")
    axes[1].set_title(f"{arch}: epoch loss")
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("mean MSE loss")
    axes[1].set_yscale("log")
    axes[2].set_title(f"{arch}: learning rate")
    axes[2].set_xlabel("epoch")
    axes[2].set_ylabel("LR")
    axes[2].set_yscale("log")
    for ax in axes:
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)
    fig.tight_layout()
    out = OUTPUT_DIR / f"{arch}_training_curves.png"
    fig.savefig(out)
    print("wrote", out)

## EMA Profiles

These plots show the post-hoc EMA basis profiles and target EMA profile shapes. They do **not** measure sample quality by themselves. The quality comparison comes later from P(k), histograms, SSCD, etc. evaluated for raw and synthesized EMA samples.

In [ ]:
def checkpoint_epochs(run_name: str) -> list[int]:
    root = CHECKPOINT_ROOT / f"{run_name}_checkpoints"
    epochs = []
    for p in sorted(root.glob("checkpoint-epoch-*")):
        ep = checkpoint_epoch(p)
        if ep is not None:
            epochs.append(ep)
    return epochs


def read_checkpoint_config(run_name: str) -> dict[str, Any] | None:
    ckpt = latest_checkpoint(run_name)
    if ckpt is None:
        return None
    path = ckpt / "checkpoint_config.yaml"
    if not path.exists():
        return None
    return yaml.safe_load(path.read_text())

ema_rows = []
for row in run_df.to_dict("records"):
    cfg = read_checkpoint_config(row["run_name"])
    epochs = checkpoint_epochs(row["run_name"])
    ema_rows.append({
        "run_name": row["run_name"],
        "arch": row["arch"],
        "variant": row["variant_tag"],
        "n_checkpoints": len(epochs),
        "last_checkpoint_epoch": max(epochs) if epochs else np.nan,
        "ema_sigma_rels_saved": cfg.get("ema_sigma_rels") if cfg else None,
        "ema_burn_in_saved": cfg.get("ema_burn_in") if cfg else None,
    })
ema_df = pd.DataFrame(ema_rows)
display(ema_df)

In [ ]:
try:
    from cosmodiff.optim import compute_ema_profiles
except Exception as exc:
    compute_ema_profiles = None
    print("Could not import compute_ema_profiles:", exc)

TARGET_EMA_FOR_PROFILE = 0.13
completed_for_ema = ema_df[(ema_df["n_checkpoints"] > 0) & ema_df["ema_sigma_rels_saved"].notna()]
if compute_ema_profiles is not None and len(completed_for_ema):
    selected = completed_for_ema.iloc[0]
    run_name = selected["run_name"]
    epochs = checkpoint_epochs(run_name)
    sigma_rels = tuple(selected["ema_sigma_rels_saved"])
    total_epochs = max(100, max(epochs) + 1)
    profile = compute_ema_profiles(
        sigma_rels_train=sigma_rels,
        checkpoint_epochs=[e + 1 for e in epochs],
        total_epochs=total_epochs,
        sigma_rel_target=TARGET_EMA_FOR_PROFILE,
    )
    fig, ax = plt.subplots(figsize=(9, 4))
    for ckpt_epoch, sigma_rel, weights in profile["basis"]:
        if ckpt_epoch in {max(e + 1 for e in epochs), min(e + 1 for e in epochs)}:
            ax.plot(profile["epochs"], weights, alpha=0.25, lw=1, label=f"basis ckpt={ckpt_epoch} sigma={sigma_rel}")
    ax.plot(profile["epochs"], profile["target"], color="black", lw=2.5, label=f"target sigma={TARGET_EMA_FOR_PROFILE}")
    ax.plot(profile["epochs"], profile["synthesized"], color="tab:red", ls="--", lw=2, label="synthesized")
    ax.set_title(f"Post-hoc EMA profile reconstruction: {run_name}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("normalized weight")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    out = OUTPUT_DIR / "ema_profile_reconstruction.png"
    fig.savefig(out)
    print("wrote", out)
else:
    print("No completed EMA checkpoints available yet.")

## Sampling Status

The sampling array writes one `.npy` per run and EMA target. This table lets you see which targets exist. The default sample file names are:

`results/nf_sweep/samples/{run_name}_seed123_{raw|ema0p03|ema0p05|ema0p08|ema0p13|ema0p20}.npy`

In [ ]:
sample_rows = []
for row in run_df.to_dict("records"):
    for label in EMA_LABELS:
        path = sample_path(row["run_name"], label)
        shape = sample_shape(path) if path.exists() else None
        sample_rows.append({
            "run_name": row["run_name"],
            "arch": row["arch"],
            "variant": row["variant_tag"],
            "ema_label": label,
            "ema_value": EMA_VALUES[label],
            "exists": path.exists(),
            "shape": shape,
            "path": str(path),
        })
sample_df = pd.DataFrame(sample_rows)
display(sample_df.pivot_table(index=["arch", "variant", "run_name"], columns="ema_label", values="exists", aggfunc="first"))
sample_status_csv = OUTPUT_DIR / "nf_sweep_sample_status.csv"
sample_df.to_csv(sample_status_csv, index=False)
print("wrote", sample_status_csv)

## Load Real And Generated Arrays

This loads real data using the run config normalization and generated `.npy` samples if they exist. By default it prefers `ema0p13` samples and falls back to `raw` or other EMA targets.

In [ ]:
def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return np.array(arr, copy=True)
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return np.array(arr[idx], copy=True)


def first_existing_sample(run_name: str, preferred: str = PREFERRED_SAMPLE_LABEL) -> tuple[str | None, Path | None]:
    order = [preferred] + [x for x in EMA_LABELS if x != preferred]
    for label in order:
        path = sample_path(run_name, label)
        if path.exists():
            return label, path
    return None, None

loaded = {}
load_rows = []
for row in run_df.to_dict("records"):
    run_name = row["run_name"]
    label, spath = first_existing_sample(run_name)
    if spath is None:
        load_rows.append({"run_name": run_name, "loaded": False, "reason": "missing generated sample"})
        continue
    config_path = Path(row["config_path"])
    if not config_path.exists():
        load_rows.append({"run_name": run_name, "loaded": False, "reason": "missing config"})
        continue
    real = load_real_from_config(config_path, max_raw_samples=MAX_RAW_REAL_CUBES)
    generated = as_nchw(load_npy(spath)).copy()
    loaded[run_name] = {"spec": row, "sample_label": label, "sample_path": spath, "real": real, "generated": generated}
    load_rows.append({
        "run_name": run_name,
        "arch": row["arch"],
        "variant": row["variant_tag"],
        "loaded": True,
        "sample_label": label,
        "real_shape": tuple(real.shape),
        "generated_shape": tuple(generated.shape),
        "sample_path": str(spath),
    })

load_df = pd.DataFrame(load_rows)
display(load_df)
print("loaded runs:", len(loaded))

In [ ]:
def plot_image_grid(images: np.ndarray, title: str, n: int = 8, cmap: str = "viridis"):
    arr = evenly_limit(as_nchw(images), min(n, len(images)))
    fig, axes = plt.subplots(1, len(arr), figsize=(1.8 * len(arr), 2.0), squeeze=False)
    for ax, img in zip(axes[0], arr):
        im = ax.imshow(img[0], origin="lower", cmap=cmap, vmin=-1, vmax=1)
        ax.set_axis_off()
    fig.suptitle(title)
    fig.tight_layout()
    return fig

# Show a compact real/generated panel for the first few loaded runs.
for run_name, bundle in list(loaded.items())[:6]:
    fig = plot_image_grid(bundle["real"], f"{run_name}: real training slices", n=8)
    fig.savefig(OUTPUT_DIR / f"{run_name}_real_grid.png")
    fig = plot_image_grid(bundle["generated"], f"{run_name}: generated ({bundle['sample_label']})", n=8)
    fig.savefig(OUTPUT_DIR / f"{run_name}_{bundle['sample_label']}_generated_grid.png")

## Histogram And P(k) Diagnostics

For P(k), the black curve is the real reference mean and the gray band is the real slice-to-slice scatter. Blue curves/points are generated samples. The ratio panel is generated mean divided by real mean.

In [ ]:
metric_rows = []
for run_name, bundle in loaded.items():
    real = evenly_limit(bundle["real"], MAX_REAL_PK)
    generated = evenly_limit(bundle["generated"], MAX_GENERATED_PK)
    hist_real = field_histogram(real)
    hist_gen = field_histogram(generated)
    pk_summary = power_spectrum_summary(real, generated, nbins=PK_NBINS)
    metric_rows.append({
        "run_name": run_name,
        "arch": bundle["spec"]["arch"],
        "variant": bundle["spec"]["variant_tag"],
        "sample_label": bundle["sample_label"],
        "generated_mean": hist_gen["mean"],
        "real_mean": hist_real["mean"],
        "generated_std": hist_gen["std"],
        "real_std": hist_real["std"],
        "generated_frac_abs_ge_0999": hist_gen["frac_abs_ge_0999"],
        **pk_summary,
    })
physics_df = pd.DataFrame(metric_rows)
if len(physics_df):
    physics_df = physics_df.sort_values(["arch", "variant", "sample_label"])
display(physics_df)
physics_csv = OUTPUT_DIR / "nf_sweep_physics_metrics.csv"
physics_df.to_csv(physics_csv, index=False)
print("wrote", physics_csv)

In [ ]:
for arch in sorted({bundle["spec"]["arch"] for bundle in loaded.values()}):
    items = [(k, v) for k, v in loaded.items() if v["spec"]["arch"] == arch]
    if not items:
        continue
    n = len(items)
    fig, axes = plt.subplots(2, n, figsize=(4.4 * n, 7.5), squeeze=False)
    for col, (run_name, bundle) in enumerate(items):
        real = evenly_limit(bundle["real"], MAX_REAL_PLOT)
        generated = evenly_limit(bundle["generated"], MAX_GENERATED_PLOT)
        real_hist = field_histogram(real)
        gen_hist = field_histogram(generated)
        edges = np.asarray(real_hist["bin_edges"])
        centers = 0.5 * (edges[:-1] + edges[1:])
        ax = axes[0, col]
        ax.plot(centers, real_hist["hist"], color="black", label="real")
        ax.plot(centers, gen_hist["hist"], color="tab:blue", label=f"generated {bundle['sample_label']}")
        ax.set_yscale("log")
        ax.set_title(bundle["spec"]["variant_tag"])
        ax.set_xlabel("normalized field value")
        ax.set_ylabel("density")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)

        real_pk, kbins = batch_power_spectra(evenly_limit(bundle["real"], MAX_REAL_PK), nbins=PK_NBINS)
        gen_pk, _ = batch_power_spectra(evenly_limit(bundle["generated"], MAX_GENERATED_PK), nbins=PK_NBINS)
        real_mean = np.nanmean(real_pk, axis=0)
        gen_mean = np.nanmean(gen_pk, axis=0)
        ratio = gen_mean / np.clip(real_mean, 1e-30, None)
        ax = axes[1, col]
        ax.plot(kbins, ratio, marker="o", color="tab:blue")
        ax.axhline(1.0, color="black", ls=":", lw=1.5)
        ax.set_xlabel("k bin")
        ax.set_ylabel("generated / real P(k)")
        ax.grid(alpha=0.25)
    fig.suptitle(f"{arch}: histogram and P(k) ratio")
    fig.tight_layout()
    out = OUTPUT_DIR / f"{arch}_hist_pk_ratio.png"
    fig.savefig(out)
    print("wrote", out)

In [ ]:
for run_name, bundle in list(loaded.items())[:8]:
    real = evenly_limit(bundle["real"], MAX_REAL_PK)
    generated = evenly_limit(bundle["generated"], MAX_GENERATED_PK)
    pk_real, kbins = batch_power_spectra(real, nbins=PK_NBINS)
    pk_gen, _ = batch_power_spectra(generated, nbins=PK_NBINS)
    real_mean = np.nanmean(pk_real, axis=0)
    real_std = np.nanstd(pk_real, axis=0)
    gen_mean = np.nanmean(pk_gen, axis=0)
    gen_std = np.nanstd(pk_gen, axis=0)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    ax = axes[0]
    ax.fill_between(kbins, np.clip(real_mean - real_std, 1e-30, None), real_mean + real_std, color="black", alpha=0.15, label="real ±1 std")
    ax.plot(kbins, real_mean, color="black", lw=2, label="real mean")
    ax.fill_between(kbins, np.clip(gen_mean - gen_std, 1e-30, None), gen_mean + gen_std, color="tab:blue", alpha=0.18, label="generated ±1 std")
    ax.plot(kbins, gen_mean, color="tab:blue", lw=2, label="generated mean")
    ax.set_yscale("log")
    ax.set_xlabel("k bin")
    ax.set_ylabel("P(k)")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8)

    ax = axes[1]
    ratio = gen_mean / np.clip(real_mean, 1e-30, None)
    ax.plot(kbins, ratio, marker="o", color="tab:blue")
    ax.axhline(1, color="black", ls=":")
    ax.set_xlabel("k bin")
    ax.set_ylabel("generated mean / real mean")
    ax.grid(alpha=0.25)
    fig.suptitle(f"{run_name} ({bundle['sample_label']})")
    fig.tight_layout()
    out = OUTPUT_DIR / f"{run_name}_{bundle['sample_label']}_pk_detail.png"
    fig.savefig(out)
    print("wrote", out)

## EMA Target Sweep Metrics

After the sampling array finishes, this compares raw and EMA-synthesized samples for each run. The first plot is P(k) mismatch versus EMA target. Lower `pk_log10_mae` is better.

In [ ]:
ema_metric_rows = []
for row in run_df.to_dict("records"):
    run_name = row["run_name"]
    config_path = Path(row["config_path"])
    if not config_path.exists():
        continue
    real = None
    for label in EMA_LABELS:
        path = sample_path(run_name, label)
        if not path.exists():
            continue
        if real is None:
            real = load_real_from_config(config_path, max_raw_samples=MAX_RAW_REAL_CUBES)
        generated = as_nchw(load_npy(path)).copy()
        pk_summary = power_spectrum_summary(
            evenly_limit(real, MAX_REAL_PK),
            evenly_limit(generated, MAX_GENERATED_PK),
            nbins=PK_NBINS,
        )
        gen_hist = field_histogram(evenly_limit(generated, MAX_GENERATED_PLOT))
        ema_metric_rows.append({
            "run_name": run_name,
            "arch": row["arch"],
            "variant": row["variant_tag"],
            "ema_label": label,
            "ema_value": EMA_VALUES[label],
            "generated_std": gen_hist["std"],
            "generated_frac_abs_ge_0999": gen_hist["frac_abs_ge_0999"],
            **pk_summary,
        })
ema_metric_df = pd.DataFrame(ema_metric_rows)
display(ema_metric_df)
ema_metric_csv = OUTPUT_DIR / "nf_sweep_ema_target_metrics.csv"
ema_metric_df.to_csv(ema_metric_csv, index=False)
print("wrote", ema_metric_csv)

In [ ]:
if len(ema_metric_df):
    for arch in sorted(ema_metric_df["arch"].dropna().unique()):
        fig, ax = plt.subplots(figsize=(8, 5))
        sub = ema_metric_df[(ema_metric_df["arch"] == arch) & ema_metric_df["ema_value"].notna()]
        for variant, g in sub.groupby("variant"):
            g = g.sort_values("ema_value")
            ax.plot(g["ema_value"], g["pk_log10_mae"], marker="o", label=variant)
        raw = ema_metric_df[(ema_metric_df["arch"] == arch) & (ema_metric_df["ema_label"] == "raw")]
        if len(raw):
            ax.scatter([0] * len(raw), raw["pk_log10_mae"], marker="x", color="black", label="raw weights")
        ax.set_title(f"{arch}: P(k) mismatch vs EMA target")
        ax.set_xlabel("EMA sigma_rel target")
        ax.set_ylabel("P(k) log10 MAE, lower is better")
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8, ncol=2)
        fig.tight_layout()
        out = OUTPUT_DIR / f"{arch}_ema_pk_sweep.png"
        fig.savefig(out)
        print("wrote", out)
else:
    print("No EMA sample metrics yet. Run the sampling array first.")

## Optional SSCD Diagnostics

Set `RUN_SSCD = True` in the configuration cell and rerun from here. SSCD measures nearest-neighbor visual similarity in an external embedding space. For each generated sample, we compute maximum similarity to the real training set. A sample above `SSCD_THRESHOLD` is counted as a near-copy.

Use CPU unless you know the notebook GPU environment has working NVRTC libraries.

In [ ]:
if RUN_SSCD:
    from simdiff_eval.sscd import load_sscd_torchscript, sscd_embeddings, sscd_generalization_metrics

    if not SSCD_PATH.exists():
        raise FileNotFoundError(
            f"Missing SSCD model: {SSCD_PATH}. Download it with the curl command from docs/sscd_generalizability_note.md."
        )
    sscd_model = load_sscd_torchscript(SSCD_PATH, device=SSCD_DEVICE)
    print("loaded SSCD on", SSCD_DEVICE)

    sscd_rows = []
    for run_name, bundle in loaded.items():
        real = evenly_limit(bundle["real"], MAX_REAL_PLOT)
        generated = evenly_limit(bundle["generated"], MAX_GENERATED_PLOT)
        cache_prefix = f"{run_name}_{bundle['sample_label']}_{SSCD_RENDER_MODE}"
        gen_cache = CACHE_DIR / f"{cache_prefix}_generated.pt"
        real_cache = CACHE_DIR / f"{cache_prefix}_real.pt"
        if gen_cache.exists() and real_cache.exists():
            import torch
            gen_emb = torch.load(gen_cache, map_location="cpu")
            real_emb = torch.load(real_cache, map_location="cpu")
        else:
            import torch
            gen_emb = sscd_embeddings(generated, sscd_model, device=SSCD_DEVICE, batch_size=SSCD_BATCH_SIZE, render_mode=SSCD_RENDER_MODE)
            real_emb = sscd_embeddings(real, sscd_model, device=SSCD_DEVICE, batch_size=SSCD_BATCH_SIZE, render_mode=SSCD_RENDER_MODE)
            torch.save(gen_emb, gen_cache)
            torch.save(real_emb, real_cache)
        metrics = sscd_generalization_metrics(gen_emb, real_emb, threshold=SSCD_THRESHOLD)
        sscd_rows.append({
            "run_name": run_name,
            "arch": bundle["spec"]["arch"],
            "variant": bundle["spec"]["variant_tag"],
            "sample_label": bundle["sample_label"],
            "n_generated": len(generated),
            "n_real": len(real),
            **metrics,
        })
    sscd_df = pd.DataFrame(sscd_rows)
    display(sscd_df)
    sscd_csv = OUTPUT_DIR / "nf_sweep_sscd_metrics.csv"
    sscd_df.to_csv(sscd_csv, index=False)
    print("wrote", sscd_csv)
else:
    print("RUN_SSCD is False. Set it to True in the configuration cell to run SSCD.")

In [ ]:
if RUN_SSCD and "sscd_df" in globals() and len(sscd_df):
    fig, ax = plt.subplots(figsize=(8, 5))
    for arch, sub in sscd_df.groupby("arch"):
        ordered = sub.sort_values("copy_fraction")
        ax.scatter(ordered["copy_fraction"], ordered["generalization_score"], label=arch)
        for _, r in ordered.iterrows():
            ax.annotate(r["variant"], (r["copy_fraction"], r["generalization_score"]), fontsize=7, alpha=0.8)
    ax.set_xlabel("SSCD copy fraction, lower is better")
    ax.set_ylabel("SSCD generalization score, higher is better")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    out = OUTPUT_DIR / "nf_sweep_sscd_copy_vs_generalization.png"
    fig.savefig(out)
    print("wrote", out)

## Final Ranking Table

This joins the currently available training, physics, EMA, and SSCD metrics. Rerun after more checkpoints or samples finish.

In [ ]:
summary = metrics_df.copy()
if "physics_df" in globals() and len(physics_df):
    summary = summary.merge(
        physics_df[["run_name", "sample_label", "pk_log10_mae", "pk_ratio_low_k", "pk_ratio_mid_k", "pk_ratio_high_k", "generated_std", "generated_frac_abs_ge_0999"]],
        on="run_name",
        how="left",
    )
if "sscd_df" in globals() and RUN_SSCD and len(sscd_df):
    summary = summary.merge(
        sscd_df[["run_name", "generalization_score", "copy_fraction", "max_similarity_median", "max_similarity_q99"]],
        on="run_name",
        how="left",
    )
cols = [c for c in [
    "run_name", "arch", "variant", "n_epochs_logged", "latest_epoch_loss", "best_epoch_loss",
    "sample_label", "pk_log10_mae", "pk_ratio_low_k", "pk_ratio_mid_k", "pk_ratio_high_k",
    "generalization_score", "copy_fraction",
] if c in summary.columns]
display(summary[cols].sort_values(["arch", "pk_log10_mae"], na_position="last"))
summary_csv = OUTPUT_DIR / "nf_sweep_summary.csv"
summary.to_csv(summary_csv, index=False)
print("wrote", summary_csv)